## Two-Stage Model (Submission)

This submission uses the Two-Stage classification pipeline described in `src/TwoStage/README.md`.  
The hyperparameters below correspond to the best configuration identified via a two-step validation: first a hard-coded exploration, then Bayesian optimization.

Performance is reported on the evaluation leaderboard.

### Performance
- Metric: Macro F1
- Best score (dev split): 0.731  
- This result is close to the structural performance ceiling achievable by linear models on this dataset, as suggested by the exploratory analysis.

### Runtime
- 8m 03s

### Hardware
- Architecture: AMD64
- CPU: 8 physical cores / 16 logical cores (max 1801 MHz)
- RAM: 31.33 GB
- GPU: none (no CUDA)

### Parameters

Two-stage rules  
- MIN_RULE_SUPPORT: 30  
- MIN_RULE_PURITY: 0.926328564964768  

Text representation  
- WORD_NG_MAX: 2  
- CHAR_NG_MAX: 5  
- MIN_DF: 2  
- MAX_DF: 0.8782583211530898  

Linear classifier  
- C_VALUE: 0.64491922705094  

Numeric features  
- n_tokens  
- title_len  
- article_len  
- title_ratio  

### Notes
- The selected configuration was first validated via hard-coded search and then confirmed through Bayesian optimization.
- This setup approaches the empirical performance ceiling of linear methods, motivating the exploration of more expressive models for further gains.
- The submission file is generated by creating the output in the `data/submission` folder.
- This file contains the full submission pipeline, including preprocessing data import, rule-based filtering, and the linear classifier.


In [3]:
#Librariyes and Frameworks
import pandas as pd
import numpy as np
import re

from collections import Counter, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


In [ ]:
# Paths 

DEV_IN_PATH  = "../../data/processed/development_processed.csv"
EVAL_IN_PATH = "../../data/processed/evaluation_processed.csv"
SUB_OUT      = "../../data/submission/submission_two_stage.csv"

In [5]:
# Best Parameters 

MIN_RULE_SUPPORT = 30
MIN_RULE_PURITY  = 0.926328564964768

WORD_NG_MAX = 2
CHAR_NG_MAX = 5
MIN_DF      = 2
MAX_DF      = 0.8782583211530898
C_VALUE     = 0.64491922705094

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

In [6]:
# Load Data 
df_dev  = pd.read_csv(DEV_IN_PATH)
df_eval = pd.read_csv(EVAL_IN_PATH)

FEATURES = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]

In [7]:
# Stage 1 - Rule Mining 

def tokenize_for_rules(text):
	# stabel per HTML / URL / boilerplate
	if not isinstance(text, str):
		return []
	return re.findall(r"[a-z0-9_:/\.]+", text.lower())


def mine_pure_rules(texts, labels):
	counts = defaultdict(lambda: Counter())

	for txt, y in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][int(y)] += 1

	rule_token_to_class = {}
	rule_meta = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < MIN_RULE_SUPPORT:
			continue

		best_class, best_freq = c.most_common(1)[0]
		purity = best_freq / total

		if purity >= MIN_RULE_PURITY:
			rule_token_to_class[tok] = int(best_class)
			rule_meta[tok] = (purity, total)

	return rule_token_to_class, rule_meta



In [8]:
# Apply  Rules 

def apply_rules(texts, rule_token_to_class, rule_meta):
	rule_pred = np.full(len(texts), -1, dtype=int)

	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rule_token_to_class]
		if not hits:
			continue

		hits.sort(
			key=lambda t: (rule_meta[t][0], rule_meta[t][1]),
			reverse=True
		)

		rule_pred[i] = rule_token_to_class[hits[0]]

	return rule_pred


In [9]:
# Stage 2 - ML Model

def make_model():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf),
	])

In [10]:
# Train 
model = make_model()
model.fit(X_dev, y_dev)

rule_token_to_class, rule_meta = mine_pure_rules(
	df_dev["article"],
	y_dev
)

print("Rules mined:", len(rule_token_to_class))

Rules mined: 168


In [11]:
# Predict Two-Stage

model_pred = model.predict(X_eval)

rule_pred = apply_rules(
	df_eval["article"],
	rule_token_to_class,
	rule_meta
)

final_pred = model_pred.copy()
mask = rule_pred != -1
final_pred[mask] = rule_pred[mask]

print(f"Rule coverage on eval: {mask.mean():.4f}")

Rule coverage on eval: 0.1808


In [13]:
# Submission
submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": final_pred.astype(int)
})

submission.to_csv(SUB_OUT, index=False)
print("Saved two-stage submission to:", SUB_OUT)

Saved two-stage submission to: ../../data/submission/submission_two_stage_4.csv
